## 개요

1. cust_number에서 9자리 ID는 제로필 해서 10자리 ID로 만들기. CCXXXX 형식은 냅두기
2. name_customer은 일단 raw로 냅두기.
3. 중복인 행 지우기 ( 다른 19개 칼럼들도 전부 포함한 채 비교해서 모든 칼럼이 중복인 행들)
4. clean_date에 있는 결측치는 NaN으로 냅두기.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset.csv").exists():
            return candidate
    raise FileNotFoundError("dataset.csv를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_PATH = PROJECT_ROOT / "dataset.csv"
OUTPUT_PATH = PROJECT_ROOT / "dataset_cleaned.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)

PROJECT_ROOT: /content
INPUT_PATH: /content/dataset.csv
OUTPUT_PATH: /content/dataset_cleaned.csv


## 1. 원본 데이터 로드



In [2]:
df_raw = pd.read_csv(INPUT_PATH, dtype=str)

print("행 수:", len(df_raw))
print("컬럼 수:", len(df_raw.columns))
print("컬럼 목록:")
print(list(df_raw.columns))

display(df_raw.head())

행 수: 50000
컬럼 수: 19
컬럼 목록:
['business_code', 'cust_number', 'name_customer', 'clear_date', 'buisness_year', 'doc_id', 'posting_date', 'document_create_date', 'document_create_date.1', 'due_in_date', 'invoice_currency', 'document type', 'posting_id', 'area_business', 'total_open_amount', 'baseline_create_date', 'cust_payment_terms', 'invoice_id', 'isOpen']


,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1930438491.0,2020-01-26,20200125,20200126,20200210.0,USD,RV,1.0,NaN,54273.28,20200126.0,NAH4,1930438491.0,0
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1929646410.0,2019-07-22,20190722,20190722,20190811.0,USD,RV,1.0,NaN,79656.6,20190722.0,NAD1,1929646410.0,0
2,U001,0200792734,MDV/ trust,2019-12-30 00:00:00,2019.0,1929873765.0,2019-09-14,20190914,20190914,20190929.0,USD,RV,1.0,NaN,2253.86,20190914.0,NAA8,1929873765.0,0
3,CA02,0140105686,SYSC llc,NaN,2020.0,2960623488.0,2020-03-30,20200330,20200330,20200410.0,CAD,RV,1.0,NaN,3299.7,20200331.0,CA10,2960623488.0,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25 00:00:00,2019.0,1930147974.0,2019-11-13,20191113,20191113,20191128.0,USD,RV,1.0,NaN,33133.29,20191113.0,NAH4,1930147974.0,0


## 2. 클렌징 전 상태 점검

In [3]:
cust_number_stripped = df_raw["cust_number"].str.strip()
is_9_digit = cust_number_stripped.str.fullmatch(r"\d{9}", na=False)
is_10_digit = cust_number_stripped.str.fullmatch(r"\d{10}", na=False)
is_other = ~(is_9_digit | is_10_digit) & cust_number_stripped.notna()

precheck = pd.DataFrame(
    {
        "metric": [
            "input_rows",
            "cust_number 9자리",
            "cust_number 10자리",
            "cust_number 기타",
            "cust_number 결측",
            "clear_date 결측",
            "원본 중복",
        ],
        "value": [
            len(df_raw),
            int(is_9_digit.sum()),
            int(is_10_digit.sum()),
            int(is_other.sum()),
            int(cust_number_stripped.isna().sum()),
            int(df_raw["clear_date"].isna().sum()),
            int(df_raw.duplicated(keep="first").sum()),
        ],
    }
)

display(precheck)

,metric,value
0,input_rows,50000
1,cust_number 9자리,2976
2,cust_number 10자리,45603
3,cust_number 기타,1421
4,cust_number 결측,0
5,clear_date 결측,10000
6,원본 중복,1161


## 3. cust_number 표준화

In [4]:
df_clean = df_raw.copy()
original_name_customer = df_raw["name_customer"].copy()

df_clean["cust_number"] = cust_number_stripped.where(
    ~is_9_digit,
    cust_number_stripped.str.zfill(10),
)

conversion_preview = pd.DataFrame(
    {
        "before": cust_number_stripped[is_9_digit].head(10).tolist(),
        "after": df_clean.loc[is_9_digit, "cust_number"].head(10).tolist(),
    }
)

display(conversion_preview)

,before,after
0,200714710,0200714710
1,200561861,0200561861
2,140105785,0140105785
3,200769623,0200769623
4,200705742,0200705742
5,200769623,0200769623
6,200780383,0200780383
7,200020431,0200020431
8,200592182,0200592182
9,200416837,0200416837


## 6. 저장


In [6]:
df_final = df_clean
df_final.to_csv(OUTPUT_PATH, index=False)

print("저장 완료:")
print(OUTPUT_PATH)

저장 완료:
/content/dataset_cleaned.csv
